# 第 7 周 - 笔记本 2：测试基座模型

## 练习目标（理念）

在微调 **之前** 先测一遍基座 **Llama 3.2**，建立误差基线（baseline），后面才能证明微调「真的变好了」：

1. 用 **4-bit 量化（BitsAndBytes / QLoRA 同款配置）** 加载基座模型
2. 在少量商品样本上试推理
3. 在测试集上算平均绝对误差等指标

**预期结果：** 大约 **$110** 的误差（很差——这正是微调的动机）

## 怎么跑

1. **需要 GPU**；本机没有就放到 Google Colab
2. 准备 `.env` 的 `HF_TOKEN`，并保证能访问 `config.BASE_MODEL` / 数据集
3. 预计 **5–10 分钟**（完整评估视 `EVAL_SIZE` 与 GPU 而定）


In [ ]:
# ========== 导入与鉴权：基座推理所需依赖 ==========

# 导入标准库 sys：修改模块搜索路径
import sys
# notebooks/ 的上一级加入 path，以便 import src
sys.path.append('..')

# 导入标准库 os：读取 HF_TOKEN 等环境变量
import os
# 导入 torch：检测 CUDA、管理张量设备
import torch
# 从 dotenv 导入 load_dotenv：加载 .env 密钥
from dotenv import load_dotenv
# 从 huggingface_hub 导入 login：登录 Hugging Face Hub
from huggingface_hub import login
# AutoTokenizer：分词；AutoModelForCausalLM：因果语言模型；BitsAndBytesConfig：4-bit 量化配置
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Item：商品样本与 prompt；evaluate：批量评估预测误差；config：全局配置
from src.items import Item
from src.evaluator import evaluate
from src.config import config

# 加载 .env 到进程环境
load_dotenv()
# 读取 Hugging Face token
hf_token = os.environ['HF_TOKEN']
# 登录 Hub（复用 git credential）
login(hf_token, add_to_git_credential=True)

# 确认环境；并打印 GPU 是否可用及设备名
print("✅ Environment loaded")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 加载测试数据

只取 **test** 子集做基线评估；train/val 用 `_` 丢弃。


In [ ]:
# 打印测试数据来源数据集名
print(f"Loading test data from: {config.DATASET_NAME}")
# from_hub 返回三元组；这里只要 test
_, _, test = Item.from_hub(config.DATASET_NAME)

# 打印条数与第一条样例，确认数据长什么样
print(f"✅ Loaded {len(test):,} test items")
print(f"\nExample item:")
print(f"  Title: {test[0].title}")
print(f"  Price: ${test[0].price:.2f}")


## 加载基座模型（4-bit 量化）

用 **NF4 + double quant** 把大模型塞进消费级 GPU，显存占用大约几个 GB 量级。


In [ ]:
# 配置 4-bit 量化（与后续 QLoRA 常用设置一致）
bnb_config = BitsAndBytesConfig(
    # 启用 4-bit 权重加载
    load_in_4bit=True,
    # 量化类型：nf4（NormalFloat4）
    bnb_4bit_quant_type="nf4",
    # 计算时用 float16，平衡速度与精度
    bnb_4bit_compute_dtype=torch.float16,
    # 双重量化：进一步压低显存
    bnb_4bit_use_double_quant=True,
)

# 提示正在加载的基座模型 id
print(f"Loading base model: {config.BASE_MODEL}")
print("This may take a few minutes...")

# 加载与基座模型匹配的分词器
tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
# 按量化配置加载因果 LM；device_map="auto" 自动放到可用 GPU
model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 若分词器没有 pad_token：用 eos_token 顶上，避免 generate 时报错
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

# 打印加载成功与大致显存占用（字节 / 1e9 → GB）
print("✅ Model loaded in 4-bit")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")


## 对样例商品试推理

先定义 `predict_base_llama`，再对手头几条商品肉眼看预测是否离谱。


In [ ]:
def predict_base_llama(item: Item) -> str:
    """用未微调的基座 Llama 预测价格：拼 prompt → generate → 按 PREFIX 截取 completion。"""
    # 若还没有 prompt：现场构造（测试用精确价格，do_round=False）
    if not item.prompt:
        item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)

    # test_prompt()：取出推理时喂给模型的文本（通常不含答案）
    prompt = item.test_prompt()

    # 分词成张量，并搬到模型所在 device（GPU/CPU）
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 推理阶段关掉梯度，省显存、加速
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            # 价格很短，只生成少量新 token
            max_new_tokens=10,
            temperature=0.1,
            # 不采样：更接近贪心/确定性输出
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    # 把整段（含 prompt）解码回字符串
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 按 config.PREFIX 切开，取最后一段作为「模型补全的价格文本」
    completion = response.split(config.PREFIX)[-1].strip()

    return completion


In [ ]:
# 先在 5 条样本上快速目检，避免一上来就跑完整评估
print("Testing base model on 5 sample products:\n")

# 依次取 test[0]..test[4]
for i in range(5):
    item = test[i]
    # 调用上面的预测函数
    prediction = predict_base_llama(item)

    # 对比标题、真实价、模型输出（文案保持原样）
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: {prediction}")
    print("-" * 60)


## 对测试集全面评估

用 `evaluate` 在 `config.EVAL_SIZE` 条上算平均误差 / MSE / R²，得到正式基线数字。


In [ ]:
# 在测试集上评估：传入预测函数、数据、样本量；workers=1 避免多进程抢同一块 GPU
results = evaluate(
    predict_base_llama,
    test,
    size=config.EVAL_SIZE,
    workers=1  # Sequential for GPU
)

# 打印基线指标块（标签文案保持英文原样）
print(f"\n{'='*60}")
print("BASE MODEL RESULTS")
print(f"{'='*60}")
print(f"Average Error: ${results['average_error']:.2f}")
print(f"MSE: {results['mse']:,.0f}")
print(f"R²: {results['r2']:.1f}%")
print(f"{'='*60}")


## 小结

✅ **基座模型测试完成！**

**预期结果：**

- Base Llama 3.2（4-bit）：大约 **$110** 误差（很差）
- 模型还不懂「怎么给商品定价」
- 预测接近随机猜测

**为什么这么差？**

- 基座模型没为价格预测做过监督训练
- 缺少本任务的产品定价领域知识
- 所以需要针对本任务做微调

**下一步：** `03_finetune_colab.ipynb` / Colab 微调笔记本 —— 用 QLoRA 把误差打到大约 **$40** 量级。
